In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import tensorflow as tf
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import os, sys, gc, time, warnings, pickle, psutil, random
from tqdm import tqdm
from math import ceil
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import math
from statsmodels.tsa.seasonal import STL
from scipy import stats
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
cal = pd.read_parquet('done_cal.parquet')
# Đọc dữ liệu
grid_df = pd.read_parquet('../data/train_val1.parquet')
# grid_df = grid_df.drop(['date'], axis=1)
grid_df = grid_df.dropna(subset=['demand'])
# print(grid_df.columns)
duplicate_id_list = grid_df[grid_df.duplicated('id', keep=False)]['id'].unique()
# print(len(duplicate_id_list))

In [10]:
duplicate_id_list1 = duplicate_id_list[100:200]
# print(duplicate_id_list1)
cnt = 0
for instance in duplicate_id_list1:
  cnt += 1
  if cnt % 100 == 0:
    break
  filtered_df = grid_df[grid_df.id == instance]

  sales_ts = pd.DataFrame({
     'timestamps': filtered_df['d'].values,
     'demand': filtered_df['demand'].values
  })
  sales_ts['d'] = sales_ts['timestamps'] # keep sales_ts['d'] as integer

  # Merge với calendar_df theo cột 'd'
  sales_ts = sales_ts.merge(cal[['d', 'date']], on='d', how='left')

  sales_ts['timestamps'] = sales_ts['date']
  sales_ts = sales_ts.drop(columns=['date'])

  sales_ts['EMA'] = sales_ts['demand'].ewm(span=2, adjust=False).mean()
  sales_ts['EMA'] = sales_ts['EMA'].apply(math.ceil)

  from statsmodels.tsa.seasonal import STL
  from scipy import stats
  import numpy as np

  # Phân tích chuỗi theo mùa vụ
  stl = STL(sales_ts['demand'], period=30)  # nếu data theo ngày, 7 là 1 tuần
  res = stl.fit()

  residual = res.resid

  # Áp dụng z-score để tìm outlier trong residual
  z = np.abs(stats.zscore(residual))
  threshold = 2.8  # ngưỡng z-score
  outliers = sales_ts[z > threshold]
  # print(outliers.head())

  for idx, row in outliers.iterrows():
       grid_df.loc[(grid_df.id == instance) & (grid_df['d'] == row['d']), 'demand'] = row['EMA']

grid_df.to_parquet('../data/train_val1.parquet')